EDA for Title boundary dataset.

What does the data in the Title boundary dataset, look like ?

These are the columns, according to https://digital-land.github.io/specification/dataset/title-boundary/

| Field | Name | Datatype | Cardinality |
|---|---|---|---:|
| `entity` | Entity | integer | 1 |
| `geometry` | Geometry | multipolygon | 1 |
| `name` | Name | string | 1 |
| `notes` | Notes | text | 1 |
| `organisation` | Organisation | curie | 1 |
| `point` | Point | point | 1 |
| `prefix` | CURIE prefix | string | 1 |
| `reference` | Reference | string | 1 |
| `entry-date` | Entry date | datetime | 1 |
| `start-date` | Start date | datetime | 1 |
| `end-date` | End date | datetime | 1 |

What is entity ? <br>
Geometry looks like polygon for a parcel. <br>
What could be the size of this dataset ?  <br>


# Looking at a record on the website
enitiy seems to be an integer , eg 14725643. <br>
All other attributes give more information about this entity. <br>
Geometry seems to be of type `MULTIPOLYGON ` <br>
Point looks like a lat and long value. <br>
Data for this entity is available as a JSON or GeoJson.
No mention of who owns this parcel. <br>
In Quality column it says -	Authoritative: We have some data from the authoritative source. What does it mean? <br>

# Checking some data in this table.

In [ ]:
import httpx
import matplotlib.pyplot as plt

color_pal = plt.rcParams['axes.prop_cycle'].by_key()['color']
plt.style.use('ggplot')

In [ ]:


base_url = "https://www.planning.data.gov.uk/entity.json"
response = httpx.get(base_url, params={"dataset": "title-boundary", "limit": 5}, timeout=60.0)
data = response.json()

In [ ]:
print(data.keys())

In [ ]:
data['entities'][0]

In [ ]:
import polars as pl

df = pl.DataFrame(data["entities"])

In [ ]:
df.describe()

In [ ]:
df.null_count()

In [ ]:
data['links']

Looks like there are not any nulls reported by polars but there are a some empty strings in the data.<br>
For 'links' what does 'last' mean? <br>
Does next mean, where I should start reading from after poling for a bactch of limit 'x' ?<br>
Count , does it mean total entries in this dataset ? This does seem to match the number in the `last' KEY for this dictionary. <br>
Too much data, how to get to swindon ? The api has an option for lat and long, I can try lat and long for town centre, see what comes. <br>


In [ ]:
# TOWN CENTRE
latitude = 51.56167431916826
longitude = -1.785358672856623

base_url = "https://www.planning.data.gov.uk/entity.json"
response = httpx.get(base_url, params={"latitude": latitude, "longitude": longitude}, timeout=10.0)
data = response.json()

In [ ]:
len(data['entities'])

In [ ]:
df = pl.DataFrame(data["entities"])

In [ ]:
df.head()

In [ ]:
df.select("dataset", "entity", "name", "reference", "prefix")

In [ ]:
df.filter(pl.col("name").str.contains("Swindon")).select("dataset", "entity", "name", "reference", "prefix")

Aha! Found data for swindon.
What is the table the query is getting this data from ?` <br>
What is entity ? <br>
Looks like entity is a row in the table <br>

What is reference ? Seems like there is a range for reference sometimes, E60000299;E60000297  <br>

A quick google search for E48000038, says "E48000038 is the official Office for National Statistics (ONS) statistical geography code for the Wiltshire & Swindon Local Resilience Forum boundary in England"
Seems like entity is just an ID for a row. <br>

More googling - 

E48000038 is the official UK statistical geography code designating the Wiltshire & Swindon Local Resilience Forum boundary <br>

E60000297 is the unique ONS (Office for National Statistics) geographic area code for the Swindon Local Planning Authority (LPA) in England. <br>

E63011979 is the official UK Office for National Statistics (ONS) Geographical Code used to identify the Swindon Built-up Area (BUA) <br>

E60000297 is the official UK government statistical identifier code for the Swindon Local Planning Authority (LPA) <br>

E06000030 is the official Government Statistical Service (GSS) code for the Borough of Swindon (Swindon Borough Council), a unitary authority in the South West region of England


In [ ]:
# OLD TOWN HALL
latitude = 51.55251771018445 
longitude = -1.7720315689678354

base_url = "https://www.planning.data.gov.uk/entity.json"
response = httpx.get(base_url, params={"latitude": latitude, "longitude": longitude}, timeout=10.0)
data = response.json()
data['entities']

So 'entity' remains for same for another area within swindon. What about reference ?

In [ ]:
# OLD TOWN HALL

reference = 'E48000038',
base_url = "https://www.planning.data.gov.uk/entity.json"
response = httpx.get(base_url, params={"reference": reference}, timeout=10.0)
data = response.json()

In [ ]:
data

Quering by reference give the same data, seems like this is for all of swindon ?
Time to take a look at local-resilience-forum-boundary dataset on the website.

In [ ]:
base_url = "https://www.planning.data.gov.uk/entity.json"
response = httpx.get(base_url, params={"dataset": "local-resilience-forum-boundary"}, timeout=60.0)
data = response.json()

In [ ]:
len(data['entities'])

In [ ]:
df = pl.DataFrame(data["entities"])

In [ ]:
df.head()

"local-authority-district"	8600027	"Swindon"	"E06000030"	"statistical-geography" <br>

Seems like the area of interest.